# 📘 코드 해설판 (Python 초보자용)

이 노트북은 원본 `4_RAG_framework_evaluation_with_MCP.ipynb` 에 **번호(`# 1`, `# 2` …)가 붙은 코드 셀마다 한국어 해설**을 덧붙인 파일입니다.
코드 셀 바로 **위**의 `💡 [# N] 코드 설명` 칸이 해설입니다. 원본 코드와 원본 설명은 그대로입니다.

**이 노트북의 흐름**

1. **평가 방법 만들기** (# 1 ~ # 9) : LLM을 "채점관"으로 써서 답이 맞는지 판정
2. **RAG 평가하기** (# 10 ~ # 12) : 금융 질문 10개로 CRAG 점수 계산
3. **문제 원인 분석** (# 13 ~ # 17) : 기존 KG 조회 방식이 왜 틀리는지 확인
4. **MCP로 개선하기** (# 18 ~ # 30) : LLM이 **스스로 도구(API)를 골라 호출**하는 에이전트 방식으로 교체

**알아 둘 용어**

| 용어 | 쉬운 뜻 |
|---|---|
| 환각(hallucination) | LLM이 틀린 내용을 그럴듯하게 지어내는 것 |
| MCP (Model Context Protocol) | LLM이 외부 도구·데이터를 쓸 수 있게 연결하는 **표준 규격**. "AI용 USB 포트"라고 생각하면 쉽습니다 |
| MCP 서버 / 클라이언트 | 도구를 제공하는 쪽 / 도구를 가져다 쓰는 쪽 |
| 도구 호출(Tool Calling) | LLM이 "이 함수를 이 값으로 실행해 줘"라고 요청하는 기능 |
| 에이전트(Agent) | 목표를 위해 스스로 도구를 골라 여러 번 호출하며 일하는 LLM |
| `async` / `await` | 시간이 걸리는 작업(네트워크 통신 등)을 **기다리는 동안 멈추지 않게** 하는 파이썬 문법 |

#📓 RAG Framework Evaluation and Upgrade


<img src="https://i.imgur.com/fTICSCN.png">

In this exercise, you will have the opportunity to evaluate the various RAG components you have built earlier. In the previous Day 1 session, we only performed evaluation using the LLM. However, this time, we will utilize various evaluation methods commonly used in the NLP field.


### I. Evaluate RAG  
### II. Upgrade KG Query Stage with MCP
  
Okay. Now we know what we have to do for this final section.  
However, we need to know additional evaluation metric for CRAG dataset:

`Exact Accuracy`  
`Accuracy`  
`Hallucination`   
`Missing`  



## 0. New evaluation metrics for CRAG dataset

In most cases, datasets designed for specific tasks are presented along with **evaluation metrics** that can be used for performance evaluation. Similarly, the CRAG dataset provides evaluation metrics that should be used when measuring the performance of LLMs on this dataset.  

Therefore, before proceeding with the evaluation, let’s first check which evaluation methods the creators of the CRAG dataset intended to use. Specifically, we will examine the four answer classification criteria they proposed, understand how these are evaluated, and clarify what each criterion means.  

<br/>

We follow the steps below:  

#### 1. What is the new evaluation metrics for CRAG dataset?
#### 2. How to evalute RAG following new evaluation metrics?


### 1. What is the new evaluation metrics for CRAG dataset?
  
The creators of the CRAG dataset evaluated RAG based on the following four elements:

<img src="https://i.imgur.com/0hxmPdi.png">

Simply put, they classified responses that were identical to their predefined answers as the most ideal case. Responses with similar meanings but containing minor errors were classified as the next most ideal case.

The important point is that, under the CRAG dataset’s evaluation criteria, everything else is not simply classified as “incorrect.” Instead, the creators expect the LLM to admit when it does not know the answer. Incorrect answers are those containing errors, and the more these answers occur, the worse the model’s performance is considered. However, answers classified as Missing (indicating no answer) do not negatively or positively impact the model’s performance.

By understanding these four classification criteria, you will gain valuable insights when analyzing evaluation results on the CRAG dataset.


### 2. How to evalute RAG following new evaluation metrics?
  
Since the evaluation criteria mentioned above cannot be measured automatically, the evaluation must be conducted using an LLM. This can be done using methods similar to Trulens. However, as the evaluation results can vary depending on the prompt used, we will use the default prompt provided by the creators of the CRAG dataset.

The evaluation prompt is as follows. Based on its content, we need to provide the LLM with the `question`, `model prediction`, and `ground truth answers`. The LLM will then generate a response by performing the evaluation according to the instructions.

To help us understand this with a simple example, please install and import the following library:

### 💡 [# 1] 코드 설명 — 라이브러리 설치

**한 줄 요약:** 평가와 MCP 실습에 필요한 라이브러리를 설치합니다.

- `os.environ.pop("PYTHONPATH", None)` : 라이브러리 충돌 방지
- `!pip install ...` : 터미널 설치 명령
  - `openai==1.81.0` : 앞 노트북보다 **새 버전**을 씁니다(MCP 관련 기능 호환을 위해)
  - `llama-index-tools-mcp` : LlamaIndex에서 MCP 서버의 도구를 쓰게 해 주는 꾸러미
  - `tqdm` : 반복문 진행률 막대 표시

> 🔰 앞 노트북과 같은 환경에서 실행했다면 버전이 바뀌므로, 설치 후 **커널을 재시작**하는 것이 안전합니다.

In [ ]:
# 1
import os
os.environ.pop("PYTHONPATH", None)

!pip install openai==1.81.0 \
    llama-index llama-index-embeddings-huggingface==0.4.0 llama-index-tools-mcp \
    packaging>=24.0 langchain nltk>=3.8.1 \
    streamlit==1.35.0 watchdog kubernetes==26.1.0 \
    blingfire beautifulsoup4 sentence-transformers ray scikit-learn \
    tqdm tiktoken transformers --quiet

### 💡 [# 2] 코드 설명 — 채점용 지시문

**한 줄 요약:** LLM에게 "모델의 답이 정답과 일치하는지 판정하라"고 시키는 지시문을 만듭니다.

채점 규칙:
1. 모델의 답이 정답 목록 중 하나와 일치하면 `"Accuracy": "True"`, 아니면 `"False"`
2. 모델이 "모르겠다/정보가 부족하다"고 답하면 항상 `"False"`
3. 정답이 "invalid question"(잘못된 질문)이면, 모델도 정확히 "invalid question"이라고 해야 `"True"`
4. 출력은 `Accuracy` 항목 하나만 있는 JSON

- `import sys` : 시스템 관련 기본 도구(여기서는 불러오기만 함)

> 🔰 "2016"과 "2016년에 데뷔했습니다"처럼 **표현만 다른 답**은 글자 비교로는 채점할 수 없습니다. 그래서 LLM에게 채점을 맡깁니다(LLM-as-a-Judge).

In [ ]:
# 2
### YOUR CODE HERE ###

INSTRUCTIONS = """
# Task:
You are given a Question, a model Prediction, and a list of Ground Truth answers, judge whether the model Prediction matches any answer from the list of Ground Truth answers. Follow the instructions step by step to make a judgement.
1. If the model prediction matches any provided answers from the Ground Truth Answer list, "Accuracy" should be "True"; otherwise, "Accuracy" should be "False".
2. If the model prediction says that it couldn't answer the question or it doesn't have enough information, "Accuracy" should always be "False".
3. If the Ground Truth is "invalid question", "Accuracy" is "True" only if the model prediction is exactly "invalid question".
# Output:
Respond with only a single JSON string with an "Accuracy" field which is "True" or "False".
"""

import sys

You can drag-and-drop the import code file into the workspace. This will allow you to import the necessary functions from that file for this practice. However, please ensure that the file is in the same folder as the currently running code for the import to succeed.  


### 💡 [# 3] 코드 설명 — 키 등록과 기존 코드 불러오기

**한 줄 요약:** API 키를 등록하고, 앞 노트북에서 만든 클래스들을 파일에서 불러옵니다.

- `os.environ["OPENAI_API_KEY"] = "sk-..."` : ⚠️ 본인 키로 교체
- `import random` : 무작위 선택 도구
- `from tqdm import tqdm` : 진행률 표시 도구
- `from import_function import LlamaIndexRetriever, Reader, KGQueryEngine` : 같은 폴더의 **`import_function.py`** 파일에 들어 있는 클래스 3개를 가져옵니다.

> 🔰 내가 만든 `.py` 파일도 라이브러리처럼 `import` 할 수 있습니다. 단, 그 파일이 **노트북과 같은 폴더**에 있어야 합니다. 앞에서 만든 코드를 다시 복사해 붙이지 않아도 되는 이유입니다.

In [ ]:
# 3
### YOUR CODE HERE ###

import os
os.environ["OPENAI_API_KEY"] = "sk-..." #Insert your openai api key

import openai
import json
import random
import bz2
from tqdm import tqdm
from import_function import LlamaIndexRetriever, Reader, KGQueryEngine

Now, to proceed with the evaluation, let’s make the dataset accessible. Run the code below. Depending on your computer environment, this may take a little time.  

### 💡 [# 4] 코드 설명 — 데이터 불러오기 (앞쪽 약 500건)

**한 줄 요약:** 데이터 파일에서 처음 501건만 읽어 `dataset` 리스트에 담습니다.

- `file_path = '/path/to/...'` : ⚠️ **실제 데이터 파일 경로로 수정**
- `bz2.open(file_path, 'rt')` : 압축 파일을 글자 모드로 열기
- `json.loads(line.strip())` : 한 줄 → 사전
- `if len(dataset) > 500: break` : 500건을 넘으면 중단(시간·메모리 절약)
- `try / except json.JSONDecodeError` : 형식이 깨진 줄이 있어도 멈추지 않고 오류만 출력

In [ ]:
# 4
### YOUR CODE HERE ###

file_path = '/path/to/CRAG dataset/crag_task_1_dev_v4_release.jsonl.bz2'

dataset = []

with bz2.open(file_path, 'rt') as file:
    for line in file:
        try:
            data = json.loads(line.strip())
            dataset.append(data)
            if len(dataset) > 500:
              break
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

Please note that, depending on your computer environment, this process might take some time.  

<img src="https://i.gifer.com/B6Qs.gif" width="150">

Thank you for your understanding.


Next, to obtain the model prediction by asking the LLM a question, we will use the following simple code to generate a response:


### 💡 [# 5] 코드 설명 — LLM 호출 함수 (역할 지정 가능)

**한 줄 요약:** 사용자 메시지와 시스템 지시문을 받아 LLM의 답을 돌려주는 함수입니다.

- `def generate_answer(user_prompt, system_prompt = "You are a helpful assistant."):` : 두 번째 매개변수에 **기본값**이 있습니다. 호출할 때 생략하면 이 기본값이 쓰입니다.
- `messages` : system(역할 지시) + user(질문)
- `openai.chat.completions.create(...)` → `.choices[0].message.content` : 답변 글 반환

> 🔰 같은 함수를 **답변 생성**에도(기본 지시문), **채점**에도(# 2의 지시문) 쓸 수 있게 만든 것입니다.

In [ ]:
# 5
### YOUR CODE HERE ###

def generate_answer(user_prompt, system_prompt = "You are a helpful assistant."):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
    )

    return response.choices[0].message.content

Now, let’s proceed with the evaluation. The evaluation will be conducted using randomly selected data points.

### 💡 [# 6] 코드 설명 — 무작위 질문 하나에 LLM이 답하게 하기

**한 줄 요약:** 데이터에서 무작위로 한 건을 뽑아 LLM(검색 없이)의 답을 받습니다.

- `random.choice(dataset)` : 리스트에서 무작위로 하나 선택 (실행할 때마다 달라짐)
- `random_data['query']` : 질문 / `['answer']` : 정답 / `['alt_ans']` : **다른 표현의 정답들**(리스트)
- `all_answers = [test_answer] + test_answer_candidate` : 리스트끼리 `+` 하면 이어 붙여집니다 → 인정되는 정답 전체 목록
- `generate_answer(user_prompt=test_question)` : `이름=값` 형태로 인자를 넘기는 방식(키워드 인자)

In [ ]:
# 6
### YOUR CODE HERE ###

random_data = random.choice(dataset)

test_question = random_data['query']
test_answer = random_data['answer']
test_answer_candidate = random_data['alt_ans']
all_answers = [test_answer] + test_answer_candidate

model_prediction = generate_answer(user_prompt=test_question)

### 💡 [# 7] 코드 설명 — LLM에게 채점시키기

**한 줄 요약:** 질문·정답 목록·모델의 답을 한 글로 묶어 채점용 LLM에 보내고 판정을 받습니다.

- `context_template = f"""..."""` : 세 가지 정보를 끼워 넣은 여러 줄 문자열
- `generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)` : 이번에는 시스템 지시문으로 # 2의 채점 규칙을 사용
- 결과는 `{"Accuracy": "True"}` 또는 `{"Accuracy": "False"}` 형태의 글

In [ ]:
# 7
### YOUR CODE HERE ###

context_template = f"""Question: {test_question}
List of Ground Truth answers: {all_answers}
Model Prediction: {model_prediction}
"""

print(context_template)

evaluation_result = generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)

print(evaluation_result)

Once the LLM evaluation results are generated, we need a function to extract the relevant results. Using the parser below, we can extract the classification results effectively:


### 💡 [# 8] 코드 설명 — 채점 결과를 숫자로 바꾸기

**한 줄 요약:** 채점 LLM의 글(JSON)을 해석해 맞으면 `1`, 그 외에는 `-1` 을 돌려주는 함수입니다.

- `response.lower()` : 전부 소문자로 (`"Accuracy"` → `"accuracy"`, `"True"` → `"true"`)
- `json.loads(response)` : 글 → 사전
- 조건: `"accuracy"` 항목이 있고, 그 값이 진짜 `True` 이거나 글자 `"true"` 이면 → `answer = 1`
  - `isinstance(값, str)` : 값이 글자인지 확인
  - `and` 는 "둘 다 참", `or` 는 "둘 중 하나라도 참"
- 그렇지 않으면 `raise ValueError(...)` 로 **일부러 오류를 발생**시킵니다.
- `except: return -1` : 오류가 나면(틀렸거나, JSON 해석 실패) `-1` 반환
- 아래 `test_text` 는 함수가 잘 동작하는지 보는 시험용 입력 → `1` 이 출력되어야 합니다.

In [ ]:
# 8
### YOUR CODE HERE ###

def parse_response(response):
    try:
        response = response.lower()
        model_resp = json.loads(response)
        answer = -1
        if "accuracy" in model_resp and (
            (
              model_resp["accuracy"] is True
            )
            or
            (
                isinstance(model_resp["accuracy"], str)
                and model_resp["accuracy"].lower() == "true"
            )
        ):
            answer = 1
        else:
            raise ValueError(f"Could not parse answer from response: {model_resp}")

        return answer
    except:
        return -1

test_text = """{
    "Accuracy": "True"
}"""

print(parse_response(test_text))

Here is the function that processes the LLM-generated response, converts it into JSON format, and checks for the accuracy attribute. If accuracy is true, it returns 1; otherwise, it returns -1.

This function can be used to perform the evaluation.

### 💡 [# 9] 코드 설명 — 채점 과정을 함수 하나로

**한 줄 요약:** # 7(채점 요청)과 # 8(결과 해석)을 묶은 `CRAG_evaluation` 함수를 만듭니다.

- 입력: 질문, 정답, 모델의 답
- 처리: 채점용 글 만들기 → 채점 LLM 호출 → `parse_response` 로 1 / -1 변환
- 출력: `1`(맞음) 또는 `-1`(틀림)

In [ ]:
# 9
### YOUR CODE HERE ###

def CRAG_evaluation(question, ground_truth, prediction):
  context_template = f"""Question: {question}
  List of Ground Truth answers: {ground_truth}
  Model Prediction: {prediction}
  """

  evaluation_result = generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)

  eval_res = parse_response(evaluation_result)

  return eval_res

## I. Evaluate RAG

So far, we have evaluated the performance of the retriever to determine which retriever can be more effective. While using the recall score as the evaluation metric makes it difficult to measure semantic similarity, we have seen that it is convenient for large-scale automatic evaluation.

In this section, we aim to evaluate the RAG system using multiple approaches. After selecting the best retriever based on the method described above, we now need to integrate it with the Reader to build a complete RAG system and verify its overall performance.
We follow the steps below:  

#### 1. Define RAG and Evaluation Metric
#### 2. Evaluate through CRAG Evaluation Method

### 1. Define RAG
Now, we will define RAG class.

### 💡 [# 10] 코드 설명 — 평가할 RAG 정의

**한 줄 요약:** 웹 검색 결과와 KG 결과를 **모두** 참고자료로 쓰는 `RAG` 클래스를 정의하고 객체를 만듭니다.

- `external_kg_server = "http://10.2.0.165:8000"` : KG API 서버 주소(강의실 내부망. 환경에 맞게 수정)
- `__init__(self, server=None)` : 검색기, KG 엔진(서버 주소 전달), Reader 준비
- `retrieve(...)` :
  - 웹 검색 결과에서 조각 검색 + KG 조회
  - `combined_results = [kg_results]` 후 `combined_results.extend(retrieved_results)` : `extend` 는 리스트 뒤에 **다른 리스트의 내용물을 풀어서** 붙입니다 → KG 결과가 맨 앞, 웹 조각들이 뒤
- `generate_response(...)` : Reader로 답 생성
- `inference(...)` : 검색 → 생성 후 **사전** `{"retrieved_results": ..., "answer": ...}` 으로 반환
- `rag = RAG(server=external_kg_server)` : 객체 생성

In [ ]:
# 10
### YOUR CODE HERE ###

external_kg_server = "http://10.2.0.165:8000"   #need to change

class RAG:
    def __init__(self, server=None):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()

    def retrieve(self, query, search_results, topk):
        retrieved_results = self.retriever.retrieve(query, search_results, topk)

        kg_results = self.kg_query_engine.query(query)

        combined_results = [kg_results]
        combined_results.extend(retrieved_results)

        return combined_results

    def generate_response(self, query, retrieved_results):
        answer = self.reader.generate_response(query, retrieved_results)
        return answer

    def inference(self, query, search_results, topk):
        retrieved_results = self.retrieve(query, search_results, topk)
        answer = self.generate_response(query, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag = RAG(server=external_kg_server)

###2. Evaluate through CRAG Evaluation Method

This time, we will evaluate the results using the evaluation metrics proposed in the CRAG dataset. As explained at the very beginning of this session, the model’s predictions are categorized into four classes for evaluation purposes.

####1.   **Perfect**: Correctly answers the question and contains no hallucination
####2.   **Acceptable**: Provide a useful answer to the question but may contain minor errors
####3.   **Missing**: The response is "I don't know", "I'm sorry I can't find ...".
####4.   **Incorrect**: The response provides wrong or irrelevant infromation to answer the question.

The model predictions categorized above are then linearized in the following manner to evaluate the final performance of the RAG system.

<img src="https://i.imgur.com/TDQ5eI4.png">

Here, let’s proceed to evaluate the validation set using the function we just defined.

We designed our Graph RAG to handle only questions within the finance domain, so we will also select test dataset questions that belong to the finance domain.

### 💡 [# 11] 코드 설명 — 금융 질문 10개 고르기

**한 줄 요약:** 데이터에서 금융 분야 질문의 ID를 앞에서부터 10개 모읍니다.

- `data['domain'] == 'finance'` 이면 `data['interaction_id']`(데이터 고유 번호)를 리스트에 추가
- `if len(finance_test_dataset_ids) >= 10: break` : 10개가 모이면 종료

In [ ]:
# 11
### YOUR CODE HERE ###

finance_test_dataset_ids = []

for data in dataset:
    if data['domain'] == 'finance':
        finance_test_dataset_ids.append(data['interaction_id'])

    if len(finance_test_dataset_ids) >= 10:
        break

### 💡 [# 12] 코드 설명 — RAG 평가 실행과 CRAG 점수 계산

**한 줄 요약:** 금융 질문 10개를 RAG로 풀고, 답을 네 종류로 분류해 최종 점수를 계산합니다.

- `n_miss, n_correct, n_correct_exact = 0, 0, 0` : 변수 세 개를 한 줄로 0으로 초기화
- `for data in tqdm(dataset):` : `tqdm` 으로 감싸면 **진행률 막대**가 표시됩니다.
- `if data['interaction_id'] not in finance_test_dataset_ids: continue` : 고른 10개가 아니면 건너뜀
- `str(data['answer']).strip().lower()` : 정답을 글자로 바꾸고, 앞뒤 공백 제거, 소문자화 (비교를 공정하게 하기 위해)
- `rag.inference(question, web_search_results, 5)['answer'].lower()` : RAG의 답(소문자)

분류 순서:

| 조건 | 분류 | 변수 |
|---|---|---|
| 답이 정답과 **글자까지 동일** | 완전 정답 | `n_correct_exact` |
| 답에 "i don't know" 포함 | 모름(missing) | `n_miss` |
| LLM 채점 결과가 1 | 정답(표현만 다름) | `n_correct` |
| 나머지 | **환각**(틀린 답) | `n_hallucinate` |

- `n_hallucinate = 전체 - 완전정답 - 정답 - 모름` : 남은 것이 환각
- `CRAG_score = n_correct_exact + 0.5*n_correct - n_hallucinate` : 완전 정답 +1점, 채점으로 인정된 정답 +0.5점, 모름 0점, **환각 -1점**

> 🔰 핵심은 **틀리게 답하면 감점**이라는 점입니다. 모를 때는 "모른다"고 하는 것이 지어내는 것보다 낫다는 평가 철학입니다.

In [ ]:
# 12
### YOUR CODE HERE ###

n_miss, n_correct, n_correct_exact = 0, 0, 0

for data in tqdm(dataset):
  if data['interaction_id'] not in finance_test_dataset_ids:
    continue

  question = data['query']
  ground_truth_lowercase = str(data['answer']).strip().lower()
  web_search_results = data['search_results']

  prediction_lowercase = rag.inference(question, web_search_results, 5)['answer'].lower()

  if prediction_lowercase == ground_truth_lowercase:
      n_correct_exact += 1
      continue
  elif "i don't know" in prediction_lowercase:
      n_miss += 1
      continue

  acceptable = CRAG_evaluation(question, ground_truth_lowercase, prediction_lowercase)

  if acceptable == 1:
    n_correct += 1

n_hallucinate = (len(finance_test_dataset_ids) - n_correct_exact - n_correct - n_miss)

CRAG_score = n_correct_exact + 0.5*n_correct - n_hallucinate

print("\n\n")
print("Number of correct answers:", n_correct)
print("Number of exact correct answers:", n_correct_exact)
print("Number of missed answers:", n_miss)
print("Number of hallucinated answers:", n_hallucinate)
print("CRAG score:", CRAG_score)

## II. Upgrade KG Query Stage with MCP

So far, we have integrated RAG with a knowledge graph, enabling the LLM to retrieve structured knowledge through KGQueryEngine. In this approach, the retrieval was based on a decision tree to invoke API calls, and queries were constructed using the LLM.

However, is our KGQueryEngine truly reliable in practical settings? Can the retrieved results be used effectively by the LLM? More importantly, how does this method compare to the emerging MCP-based retrieval approach?

To answer these questions, we will upgrade our RAG system and evaluate both methods through the following steps:

####1. Error Case Analysis
####2. Implement MCP-based Tool Calls

This structured comparison will help us determine whether the shift toward MCP tools—now gaining popularity in LLM applications—is practically justified.

###1. Error Case Analysis

Previously, we built Graph RAG and confirmed that incorporating a knowledge graph, rather than relying solely on RAG, can improve performance for certain examples.

However, it remains uncertain whether our `KGQueryEngine` functions correctly for all question-answer pairs. We have yet to analyze all questions within the finance domain.

Beyond questions that require information such as EPS, there can be various other questions within the finance domain. For example, consider the following questions:

### 💡 [# 13] 코드 설명 — 분석할 오답 사례 불러오기

**한 줄 요약:** 특정 ID의 데이터 한 건을 찾아 질문·정답·검색 결과·질문 시각을 변수에 저장합니다.

- `if data['interaction_id'] == "7a77679e-...":` : 해당 ID를 찾으면
- `question`, `gold_answer`, `search_results`, `query_time` 저장 후 `break`

> 🔰 `query_time`(질문한 시각)이 중요합니다. "어제 주가"의 '어제'가 며칠인지는 질문 시각을 알아야 정해지기 때문입니다. **이 변수들은 뒤 셀에서 계속 사용**됩니다.

In [ ]:
# 13
### YOUR CODE HERE ###

with bz2.open(file_path, 'rt') as file:
  for line in file:
    data = json.loads(line.strip())

    if data['interaction_id'] == "7a77679e-d88b-4acf-9532-94e32233950b":
      question = data['query']
      gold_answer = data['answer']
      search_results = data['search_results']
      query_time = data['query_time']
      break

print("Question: ", question)
print("Gold Answer: ", gold_answer)
print("Query Time: ", query_time)

Let’s examine the results generated by our KGQueryEngine for this question.

To do so, we need to recall how we perform retrievals on the knowledge graph. We use an LLM to generate queries that serve as inputs for APIs connected to the knowledge graph.

However, the LLM models provided by OpenAI inherently exhibit randomness, meaning that the same query is not always generated consistently. As a result, our KGQueryEngine does not always return identical results.

Therefore, it is important to repeat the same process multiple times to identify trends in the generated outputs.

Let’s explore the irregularity of GPT and our query generation examples through the following case study.


### 💡 [# 14] 코드 설명 — 같은 질문을 3번 구조화해 보기

**한 줄 요약:** 기존 `KGQueryEngine` 으로 같은 질문을 3번 분석해 결과가 매번 같은지 확인합니다.

- `KGQueryEngine(server=external_kg_server)` : KG 엔진 생성
- `for i in range(3):` : 3번 반복
- `kg_query_engine.generate_query(question)[0]` : (구조화 결과, 금융 여부) 중 **첫 번째**만 사용
- `generated_queries.append(...)` : 결과 모으기

> 🔰 특히 `datetime` 값을 눈여겨보세요. 프롬프트에 질문 시각을 주지 않았기 때문에 LLM이 날짜를 **추측**하게 되고, 틀리거나 매번 달라질 수 있습니다.

In [ ]:
# 14
### YOUR CODE HERE ###

kg_query_engine = KGQueryEngine(server=external_kg_server)

generated_queries = []

for i in range(3):
    generated_query = kg_query_engine.generate_query(question)[0]
    print(generated_query)
    generated_queries.append(generated_query)

Select queries that explicitly specify company names, metrics, and other relevant details, and examine the results of knowledge graph retrieval


### 💡 [# 15] 코드 설명 — KG 조회 결과 살펴보기

**한 줄 요약:** 구조화된 질문 하나로 KG를 조회하고, 결과를 보기 좋게 출력한 뒤 글자 수를 확인합니다.

- `i=0` : 3개 중 몇 번째 결과를 쓸지(0, 1, 2로 바꿔 보기)
- `kg_query_engine.get_finance_kg_results(generated_queries[i])` : KG 조회 → `<DOC>` 로 이어진 긴 글
- `kg_results.split("<DOC>\n")` : 구분자로 다시 나눔
- `[s.replace("'", '"') for s in json_strings]` : 작은따옴표를 큰따옴표로 교체(JSON은 큰따옴표만 인정)
- `[json.loads(js) for js in json_strings]` : 각각을 사전으로 변환
- `json.dumps(data, indent=4)` : 들여쓰기해 출력
- 마지막 줄 `len(kg_results)` : 결과 글자 수 (셀 마지막 줄의 값은 자동으로 표시됨)

> 🔰 날짜를 못 맞추면 필터링이 안 되어 **전체 기간 데이터**가 통째로 나옵니다. 너무 길어서 Reader의 4000자 제한에 잘리고, 정작 필요한 날짜의 값이 빠질 수 있습니다.

In [ ]:
# 15
### YOUR CODE HERE ###

i=0 #change this index

kg_results = kg_query_engine.get_finance_kg_results(generated_queries[i])

json_strings = kg_results.split("<DOC>\n")

json_strings = [s.replace("'", '"') for s in json_strings]

parsed_json = [json.loads(js) for js in json_strings]

for idx, data in enumerate(parsed_json):
    print(json.dumps(data, indent=4))

len(kg_results)

Upon reviewing the search results, we observed that an extremely long string was retrieved.  

Such lengthy search results can lead to the following issues:  

1. **Excessive unnecessary information may cause hallucinations.**  
2. **Even if the necessary information is retrieved, the LLM may fail to recognize it properly.**  
3. **When combined with search results from `search_results`, the total context length may exceed the LLM’s limit, leading to inference errors.**  

Let’s analyze how our RAG responds to this issue. Here, we will focus on handling search results from the knowledge graph, excluding the search process from `search_results`.  

Therefore, let’s declare a new RAG class as follows and use it for evaluation.

### 💡 [# 16] 코드 설명 — KG만 쓰는 RAG

**한 줄 요약:** 웹 검색 결과를 빼고 KG 결과만 참고자료로 쓰는 `RAGwithoutSR` 을 정의합니다. (SR = Search Results)

- # 10의 `RAG` 와 같지만 `retrieve` 에서 `return [kg_results]` 로 **KG 결과만** 반환합니다.
- `rag_kg = RAGwithoutSR(server=external_kg_server)` : 객체 생성

> 🔰 원인을 분리해서 보기 위한 실험입니다. 웹 검색을 빼면 문제가 KG 쪽에 있는지 분명히 알 수 있습니다.

In [ ]:
# 16
### YOUR CODE HERE ###

class RAGwithoutSR:
    def __init__(self, server=None):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()

    def retrieve(self, query, search_results, topk):
        kg_results = self.kg_query_engine.query(query)

        return [kg_results]

    def generate_response(self, query, retrieved_results):
        answer = self.reader.generate_response(query, retrieved_results)
        return answer

    def inference(self, query, search_results, topk):
        retrieved_results = self.retrieve(query, search_results, topk)
        answer = self.generate_response(query, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag_kg = RAGwithoutSR(server=external_kg_server)

### 💡 [# 17] 코드 설명 — 같은 질문 3번 풀기

**한 줄 요약:** KG 전용 RAG로 같은 질문을 3번 풀어 답이 안정적인지 봅니다.

- `rag_kg.inference(question, search_results, 5)` → `rag_output['answer']` : 사전에서 답만 꺼내 출력

> 🔰 정답(`gold_answer`)과 비교해 보세요. 틀리거나 매번 다르다면, **미리 짜 둔 규칙(if/elif)으로 API를 고르는 방식**의 한계입니다. 이를 MCP로 개선합니다.

In [ ]:
# 17
### YOUR CODE HERE ###

for i in range(3):
    rag_output = rag_kg.inference(question, search_results, 5)
    print(rag_output['answer'])

###2. Implement MCP-based Tool Calls

In this step, we replace the existing KGQueryEngine logic with MCP-based tool calls.

**Model Context Protocol (MCP)** is a standardized interface that allows large language models to interact with external tools or data services via a structured client-server protocol. Each function on the server is registered as a tool, and the LLM can invoke these tools by sending structured requests through the MCP client.

<img src="https://i.imgur.com/P1g0TPh.png">

Therefore, if an external data source can be accessed by the LLM through MCP, this interaction can be viewed as a form of RAG. This is especially relevant in our scenario, where the knowledge graph is only partially accessible via APIs. In such cases, instead of relying on a pre-defined decision tree, it might be more effective to let the LLM decide which tools to use dynamically.

In other words, the LLM itself selects the appropriate tool to retrieve specific information from the knowledge graph.

To enable this, it is crucial to define which tools are available to the LLM. This is handled on the MCP server side, where each tool is registered along with a description that helps the model understand what it can do.

<img src="https://i.imgur.com/EiQfJxQ.png" width=500>

The MCP server consists of core tool definitions and server-side settings that together define the server’s behavior.  Thus, we exclude server execution from this exercise, but you can refer to the attached code for implementation details.

We can leverage the same tools we used previously, such as llamaindex, to construct the MCP client. Refer to the code below to see how the MCP client can be implemented.

Now, let’s proceed to implement the MCP client. This client is responsible for retrieving tool information from the server and forwarding tool invocation requests on behalf of the LLM.

First, let's import what to need.

### 💡 [# 18] 코드 설명 — MCP 관련 도구 불러오기

**한 줄 요약:** MCP 서버에 접속하고 에이전트를 만드는 데 필요한 도구들을 불러옵니다.

- `BasicMCPClient` : MCP 서버에 접속하는 클라이언트
- `McpToolSpec` : MCP 서버가 제공하는 도구들을 LlamaIndex용 도구로 바꿔 주는 변환기
- `FunctionAgent` : 도구를 스스로 골라 쓰는 **에이전트**
- `ToolCall`, `ToolCallResult` : "도구를 호출함" / "도구가 결과를 돌려줌"을 나타내는 이벤트 종류
- `Context` : 에이전트의 대화·작업 기록 저장소
- `import dotenv` : `.env` 파일에서 설정을 읽는 도구(여기서는 불러오기만 함)

In [ ]:
# 18
### YOUR CODE HERE ###

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult, ToolCall
from llama_index.core.workflow import Context

import dotenv

Second, apart from this, the current reader is not considering query_time. Therefore, let's use the revised version to consider query_time.

### 💡 [# 19] 코드 설명 — 질문 시각을 아는 Reader

**한 줄 요약:** 기존 Reader에 **질문 시각(`query_time`)** 을 추가로 전달하도록 고친 새 `Reader` 를 정의합니다.

- 시스템 프롬프트에 "질문, **질문이 던져진 시각**, 참고자료가 주어진다"는 내용 추가
- `generate_response(self, question, query_time, top_k_chunks)` : 매개변수에 `query_time` 추가
- `prompt_generator` : 사용자 메시지 끝에 `Query Time: ...` 한 줄 추가
- 참고자료를 4000자로 자르던 줄이 **없어졌습니다**(MCP 결과는 이미 필요한 부분만 담고 있기 때문).

> 🔰 같은 이름 `Reader` 로 다시 정의했으므로, # 3에서 불러온 예전 `Reader` 는 **덮어써집니다.**

In [ ]:
# 19
### YOUR CODE HERE ###

from openai import OpenAI

oai_client = OpenAI()

class Reader:
  def __init__(self):

    self.system_prompt = """
    You are provided with a question, the time at which the question is asked, and various references.
    Your task is to answer the question succinctly, using the fewest words possible.
    If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
    There is no need to explain the reasoning behind your answers.
    """

  def generate_response(self, question: str, query_time: str, top_k_chunks: list) -> str:
    """
    Generate answer from context.
    """
    llm_input = self.prompt_generator(question, query_time, top_k_chunks)
    completion = oai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=
    llm_input
    ).choices[0].message.content
    return completion

  def prompt_generator(self, query, query_time, top_k_chunks):
    user_message = ""
    references = ""

    if len(top_k_chunks) > 0:
        references += "# References \n"
        # Format the top sentences as references in the model's prompt template.
        for chunk_id, chunk in enumerate(top_k_chunks):
            references += f"- {chunk.strip()}\n"


    user_message += f"{references}\n------\n\n"
    user_message
    user_message += f"Using only the references listed above, answer the following question: \n"
    user_message += f"Question: {query}\n"
    user_message += f"Query Time: {query_time}\n"

    llm_input = [
    {"role": "system", "content": self.system_prompt},
    {"role": "user", "content": user_message},
    ]

    return llm_input

### MCP Server Overview

The **MCP (Model-Context-Program) Server** serves as the execution and data layer that interfaces directly with the LLM. It exposes customized functionality and structured contextual information through three main elements: **Resources**, **Tools**, and **Prompts**. This enables LLMs to make informed, context-aware decisions and perform real-world operations safely and efficiently.

---

### MCP Context Components
---

### 1. Resource

A **Resource** provides structured, external data that the LLM can reference during reasoning or decision-making. It represents external objects—such as datasets, files, graph nodes, or static configuration tables—that are read-only from the model's perspective.

**Key Properties:**
- `name`: Human-readable name shown to users or in debugging.
- `description`: A short summary explaining the resource's purpose.
- `mime_type`: The MIME type of the content (e.g., `text/plain`, `application/json`). This describes the content format.
- `uri`: A globally unique identifier for the resource.
- `content`: The actual data or a pointer to the data (e.g., a JSON object, string, or file reference).

**Usage Notes:**
- Resources are **not invoked** like functions.
- They are **fetched by URI**, and remain **static** so that the LLM can repeatedly refer to them without reloading.
- Resources improve LLM accuracy by anchoring it to reliable external context.

---

### 2. Tool

A **Tool** is a callable function registered on the MCP server that the LLM can use to interact with the outside world. These tools bridge natural language instructions and real-world effects by turning a model’s intent into executable actions.

**Typical Use Cases:**
- Executing database queries
- Fetching live data from APIs
- Performing calculations or summarizations
- Searching documents or files

**How It Works:**
1. The user provides a natural language input (e.g., “Please calculate 3 + 7”).
2. The MCP client supplies the LLM with a list of available tools and their schema.
3. The LLM chooses the appropriate tool and extracts input arguments (e.g., `x=3`, `y=7`).
4. The tool invocation is generated and passed to the MCP server.
5. The tool is executed on the server, and the result is returned.
6. The LLM incorporates the result into its context to generate a final user response.

**Implementation:**
Tools are registered using the `@mcp.tool()` decorator in Python, and must conform to the function signature and type schema required by the MCP framework.

---

### 3. Prompt

A **Prompt** in MCP serves as a reusable, templated instruction that guides the LLM’s reasoning in a structured way. Prompts help the model:

- Interpret ambiguous user queries
- Structure downstream tool invocations
- Extract relevant content from complex input

**Prompt Usage Scenarios:**
- Generate structured summaries from unstructured context
- Classify user intent
- Disambiguate time references like “this year” or “last quarter”

Prompts act as templates or procedural guides that the LLM fills in dynamically based on user input and context. While not executable like tools, they play a key role in shaping how the LLM prepares inputs and interprets outputs.

---

### Summary

| Component | Purpose |
|----------|---------|
| `Resource` | Static external data that LLM can reference |
| `Tool`     | Callable function to perform real-world actions |
| `Prompt`   | Structured, reusable guide for reasoning or template generation |

Together, these elements form the foundation of the **MCP Server**, enabling LLMs to operate safely, reliably, and contextually in real applications.

### MCP Client Overview

In this section, we focus on the **MCP Client**, which serves as the interface between the LLM runtime (e.g., LlamaIndex) and an external **MCP Server**.

### What is the MCP Client?

The MCP Client is responsible for:

- Connecting to the MCP Server over HTTP or SSE.
- Fetching available **tools**, **resources**, and **prompts** exposed by the server.
- Translating natural language requests into structured tool invocations.
- Managing context and integrating responses from the server back into the LLM workflow.

It essentially acts as a **bridge** that allows the language model to interact with external APIs and structured context in a modular and dynamic way.

### 💡 [# 20] 코드 설명 — MCP 서버 접속과 도구 목록 확인

**한 줄 요약:** MCP 서버에 연결해 어떤 도구들을 쓸 수 있는지 이름과 설명을 출력합니다.

- `external_mcp_server = "http://10.2.0.165:8081/sse"` : MCP 서버 주소(환경에 맞게 수정). `sse` 는 서버가 결과를 실시간으로 흘려보내 주는 통신 방식
- `BasicMCPClient(주소)` : 접속 객체 / `McpToolSpec(client=...)` : 도구 변환기
- `tools = await mcp_tool.to_tool_list_async()` : 서버에서 도구 목록을 받아 옴
  - **`await`** : "이 작업이 끝날 때까지 기다렸다가 결과를 받아라." 네트워크 통신처럼 시간이 걸리는 **비동기 함수** 앞에 붙입니다.
- `tool.metadata.name`, `tool.metadata.description` : 도구 이름과 설명

> 🔰 LLM은 이 **설명(description)** 을 읽고 어떤 도구를 쓸지 판단합니다. 주피터에서는 셀에 `await` 를 바로 쓸 수 있지만, 일반 `.py` 파일에서는 `async def` 함수 안에서만 쓸 수 있습니다.

In [ ]:
# 20
### YOUR CODE HERE ###

external_mcp_server = "http://10.2.0.165:8081/sse" #change to correct uri

mcp_client = BasicMCPClient(external_mcp_server)
mcp_tool = McpToolSpec(client=mcp_client)

tools = await mcp_tool.to_tool_list_async()
print("\n=== Available Tools ===\n")
for tool in tools:
    print(f"🔧 Name       : {tool.metadata.name}")
    print(f"   Description: {tool.metadata.description}\n")

### 💡 [# 21] 코드 설명 — MCP 리소스 목록 확인

**한 줄 요약:** MCP 서버가 제공하는 **리소스**(참고 자료)의 목록을 출력합니다.

- `await mcp_tool.fetch_resources()` : 리소스 목록 받기
- `resource.uri`(주소), `name`(이름), `description`(설명), `mimeType`(자료 형식) 출력

> 🔰 **도구(tool)** 는 "실행하는 기능", **리소스(resource)** 는 "읽을 수 있는 자료"입니다.

In [ ]:
# 21
resources = await mcp_tool.fetch_resources()

print("\n=== Available Resources ===\n")
for resource in resources:
    print(f"📦 URI        : {resource.uri}")
    print(f"   Name       : {resource.name}")
    print(f"   Description: {resource.description}")
    print(f"   MIME Type  : {resource.mimeType}\n")

#### 1. KG Query Engine Initialization with MCP Client

In this step, we initialize an **LLM agent** that can interact with tools registered on the MCP Server.

- `BasicMCPClient` connects to the MCP Server at the specified URL.
- `McpToolSpec` wraps available tools for the agent to use.
- `FunctionAgent` is created with:
  - The `GPT` model
  - A system prompt guiding tool-based reasoning
  - A dynamic list of MCP tools

The agent is now ready to handle user queries by calling tools exposed via the MCP interface.


### 💡 [# 22] 코드 설명 — 에이전트용 시스템 프롬프트

**한 줄 요약:** 에이전트에게 "지식그래프와 상호작용하려면 도구를 사용하라"고 알려 주는 지시문입니다.

- `"""\` : 여는 따옴표 바로 뒤의 `\` 는 첫 줄의 줄바꿈을 없애 줍니다.

In [ ]:
# 22
### YOUR CODE HERE ###

SYSTEM_PROMPT = """\
You are an AI assistant for Tool Calling.

Before you help a user, you need to work with tools to interact with Our Knowledge Graph
"""

### 💡 [# 23] 코드 설명 — MCP 기반 KG 조회 엔진

**한 줄 요약:** LLM 에이전트가 MCP 도구를 스스로 골라 호출해 KG에서 정보를 가져오는 `KGQueryEngineWithMCP` 를 정의하고 초기화합니다.

**`__init__(self, mcp_tool_spec, model, llm=None)`**
- `self.llm = llm or OpenAI(model=model, temperature=0)` : `A or B` 는 "A가 있으면 A, 없으면 B". 따로 준 LLM이 없으면 새로 만듭니다.
- `self.agent = None`, `self.agent_context = None` : 에이전트는 아직 없음(`None`)으로 표시

**`async def init_agent(self)`** — 에이전트 준비
- `async def` : **비동기 함수** 정의. 호출할 때 `await` 를 붙여야 합니다.
- `tools = await self.mcp_tool_spec.to_tool_list_async()` : 도구 목록 받기
- 도구 설명이 1000자를 넘으면 1000자로 자르고 `"..."` 를 붙임 (프롬프트가 너무 길어지는 것 방지)
- `FunctionAgent(name=..., description=..., tools=tools, llm=self.llm, system_prompt=SYSTEM_PROMPT)` : 에이전트 생성
- `Context(self.agent)` : 에이전트의 작업 기록 저장소

**`async def query(self, question, verbose=False)`** — 질문 처리
- 에이전트가 준비되지 않았으면 `raise RuntimeError(...)` 로 오류 발생
- `handler = self.agent.run(question, ctx=self.agent_context)` : 에이전트 실행 시작
- `async for event in handler.stream_events():` : 실행 중 일어나는 일(이벤트)을 **실시간으로 하나씩** 받음
  - `verbose` 가 True이면, 도구 호출(`ToolCall`) 때 "어떤 도구를 어떤 값으로 불렀는지", 결과(`ToolCallResult`) 때 "무엇을 돌려받았는지" 출력
- `response = await handler` : 최종 답이 나올 때까지 기다림

**마지막 두 줄** : `gpt-3.5-turbo` 로 엔진 객체를 만들고 `await kg_engine.init_agent()` 로 준비

> 🔰 **기존 방식과의 차이:** 예전에는 사람이 `if metric == 'price': ...` 처럼 규칙을 일일이 짰습니다. 이제는 LLM이 도구 설명을 읽고 **"회사명 찾기 → 티커 찾기 → 주가 조회"** 순서를 스스로 정해 실행합니다.
> (참고: `Optional[...]` 은 "이 타입이거나 None"이라는 타입 힌트입니다. 함수 안의 `self.x: 타입` 형태 힌트는 실행 시 계산되지 않아 `Optional` 을 import 하지 않았어도 오류가 나지 않습니다.)

In [ ]:
# 23
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class KGQueryEngineWithMCP:
    def __init__(self, mcp_tool_spec: McpToolSpec, model: str, llm = None):
        self.llm = llm or OpenAI(model=model, temperature=0)
        self.mcp_tool_spec = mcp_tool_spec
        self.agent: Optional[FunctionAgent] = None
        self.agent_context: Optional[Context] = None

    async def init_agent(self):
        tools = await self.mcp_tool_spec.to_tool_list_async()
        for tool in tools:
            if len(tool.metadata.description) > 1000:
                tool.metadata.description = tool.metadata.description[:1000] + "..."
        self.agent = FunctionAgent(
            name="Agent",
            description="An agent that can work with Our Knowledge Graph api.",
            tools=tools,
            llm=self.llm,
            system_prompt=SYSTEM_PROMPT,
        )
        self.agent_context = Context(self.agent)

    async def query(self, question: str, verbose: bool = False) -> str:
        if self.agent is None or self.agent_context is None:
            raise RuntimeError("Agent not initialized. Call `await init_agent()` first.")

        handler = self.agent.run(question, ctx=self.agent_context)
        async for event in handler.stream_events():
            if verbose and type(event) == ToolCall:
                print(f"Calling tool {event.tool_name} with kwargs {event.tool_kwargs}")
            elif verbose and type(event) == ToolCallResult:
                print(f"Tool {event.tool_name} returned {event.tool_output}")

        response = await handler
        return str(response)

kg_engine = KGQueryEngineWithMCP(mcp_tool, model='gpt-3.5-turbo')
await kg_engine.init_agent()

This function executes an LLM agent `FunctionAgent` using a given user message and shared workflow context `Context`.    
It streams intermediate tool call events in real time and returns the final response from the agent.

1. Starts the agent with the input message and context.
2. Streams events while the agent is running.
   - Logs tool invocations and results if `verbose=True`.
3. Awaits the final result and returns it as a string.

This function helps monitor the reasoning and tool execution steps taken by the agent in a transparent, asynchronous manner.

Through the code below, let's see how the MCP-based RAG is solving the 'error case seen above

### 💡 [# 24] 코드 설명 — MCP 에이전트에게 질문하기

**한 줄 요약:** # 13의 질문을 에이전트에 그대로 넣어, 어떤 도구를 호출하는지 지켜봅니다.

- `await kg_engine.query(question, verbose=True)` : `verbose=True` 라서 도구 호출 과정이 화면에 출력됩니다.

> 🔰 출력의 `Calling tool ... with kwargs ...` 줄을 보면 에이전트가 **어떤 날짜**로 조회했는지 알 수 있습니다. 질문 시각을 알려 주지 않았으므로 날짜를 잘못 추측할 가능성이 큽니다.

In [ ]:
# 24
### YOUR CODE HERE ###

print("Question: ", question)

response = await kg_engine.query(
    question,
    verbose=True,
)
print(response)

Let's examine the result of the code above.  
Did the MCP agent generate a correct answer? Probably not.

The reason becomes clear when we look at the question that was passed to the LLM.  
The LLM did not receive any information about the **query time**, which is crucial for answering this type of question.  
Since the timing context is highly important here, let’s include the query time and try the question again.

### 💡 [# 25] 코드 설명 — 질문 시각을 함께 알려 주기

**한 줄 요약:** 질문에 "질문 시각을 현재로 간주하라"는 지시와 `query_time` 을 덧붙여 다시 묻습니다.

- `"문장" + "Query: " + question + "\nQuery time: " + query_time` : `+` 로 글자들을 이어 붙여 하나의 긴 질문을 만듦
- 지시 내용: "질문에서 시간 범위를 추측하지 말고, 질문 시각을 실제 현재로 삼아 '어제' 같은 상대적 표현을 그 날짜 기준으로 해석하라"

> 🔰 # 24와 도구 호출의 날짜가 어떻게 달라졌는지 비교해 보세요. **프롬프트에 필요한 정보를 주는 것**만으로 결과가 좋아질 수 있습니다.

In [ ]:
# 25
### YOUR CODE HERE ###

question_with_time = "Do not infer the time scope from the query; use the query time as the literal current moment, and treat any relative expression as anchored to that exact date\n" + "Query: " + question + "\nQuery time: " + query_time
print("Question: ", question_with_time)

response = await kg_engine.query(
    question_with_time,
    verbose=True,
)
print(response)

Was the generated result good enough?

Some users may find it insufficient, while others may encounter outright errors.  
In most cases, these issues arise because the LLM used in MCP exceeded its **context limit** due to the large amount of input.

There are several possible solutions to this problem, but one practical approach is to switch to a model with a **larger context window**.

This time, let's proceed by using the `gpt-4o-mini` model, which supports a larger input context and is better suited for handling longer queries.

### 💡 [# 26] 코드 설명 — 더 똑똑한 모델로 교체

**한 줄 요약:** 에이전트의 LLM을 `gpt-3.5-turbo` 에서 `gpt-4o-mini` 로 바꿔 다시 준비합니다.

- `KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')` : 모델 이름만 바꿔 새 객체 생성
- `await kg_engine.init_agent()` : 에이전트 초기화

> 🔰 도구를 여러 번 순서대로 호출하는 일은 추론 능력이 필요해서, 모델 성능에 따라 결과 차이가 큽니다.

In [ ]:
# 26
### YOUR CODE HERE ###

kg_engine = KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')
await kg_engine.init_agent()

### 💡 [# 27] 코드 설명 — 새 모델로 다시 질문

**한 줄 요약:** 시각 정보가 포함된 질문에 지시를 한 번 더 강조해 새 에이전트에 묻습니다.

- `question_with_time + "\nWhen inferring time information, rely solely on the query time. ..."` : "시간은 오직 질문 시각만 근거로 판단하라"는 문장 추가
- `verbose=True` 로 도구 호출 과정 확인

> 🔰 # 25(gpt-3.5-turbo)의 결과와 비교해 보세요.

In [ ]:
# 27
### YOUR CODE HERE ###

print("Question: ", question_with_time)

response = await kg_engine.query(
    question_with_time + "\nWhen inferring time information, rely solely on the query time. Do not infer the time scope from the query itself.",
    verbose=True,
)
print(response)

### 2. Structuring a RAG Class for MCP-based Interaction

To modularize the code above and enable scalable, query-driven interaction with the MCP Server, we can refactor it into a unified `RAG` class.

### Design Motivation

- MCP-based clients require **rich, well-structured input** to maximize tool usage accuracy.
- The CRAG dataset provides us with structured components such as:
  - Query (question)
  - Context (retrieved passages)
  - Metadata (timestamps, entities, etc.)

By encapsulating this into a class, we can:
1. **Generate precise questions** from structured CRAG input
2. **Run those questions through the MCP-connected agent**
3. **Return clean, tool-integrated answers**

---

### Key Components

| Method | Purpose |
|--------|---------|
| `__init__` | Initialize MCP client, tools, and LLM agent |
| `retrieve(query, query_time)` | Convert CRAG sample into a natural-language query with max context and retrieve from knowledge graph |
| `generate_response(query, query_time, )` | Perform LLM reference based on searched results to obtain LLM response |
| `inference(query, query_time)` | Combine the above two methods into a method that can perform QATask in RAG |

### 💡 [# 28] 코드 설명 — MCP 기반 RAG 클래스

**한 줄 요약:** MCP 에이전트(정보 수집)와 새 Reader(답 생성)를 합친 `RAGwithMCP` 를 정의합니다.

- `__init__(self, mcp_tool)` : `KGQueryEngineWithMCP`(gpt-4o-mini)와 `Reader` 준비
- `async def retrieve(self, query, query_time, search_results, topk)` :
  - `await self.mcp_application.init_agent()` : 에이전트 초기화(질문마다 새 기록으로 시작)
  - `full_query = f"""Query: {query} ... Query time: {query_time}"""` : 질문 + 시간 관련 지시 + 질문 시각
  - `await self.mcp_application.query(full_query, verbose=False)` : 에이전트 실행(과정 출력 안 함)
- `generate_response(...)` : Reader로 답 생성 (일반 함수라 `await` 불필요)
- `async def inference(...)` : `retrieve` → `generate_response` 후 사전으로 반환
- `rag_mcp = RAGwithMCP(mcp_tool)` : 객체 생성

> 🔰 안에서 `await` 를 쓰는 함수는 반드시 `async def` 로 정의해야 합니다. `search_results`, `topk` 는 기존 RAG와 **형식을 맞추기 위해** 받기만 하고 실제로는 쓰지 않습니다.

In [ ]:
# 28
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class RAGwithMCP:
    def __init__(self, mcp_tool):
        self.mcp_application = KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')
        self.reader = Reader()

    async def retrieve(self, query: str, query_time: str, search_results: list, topk: int):
        await self.mcp_application.init_agent()

        full_query = f"""Query: {query} When inferring time information, rely solely on the query time. Do not infer the time scope from the query itself.
Query time: {query_time}"""

        mcp_result = await self.mcp_application.query(full_query, verbose=False)
        return mcp_result

    def generate_response(self, query: str, query_time: str, retrieved_results: str):
        answer = self.reader.generate_response(query, query_time, retrieved_results)
        return answer

    async def inference(self, query: str, search_results: list, query_time: str, topk: int):
        retrieved_results = await self.retrieve(query, query_time, search_results, topk)
        answer = self.generate_response(query, query_time, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag_mcp = RAGwithMCP(mcp_tool)

### 💡 [# 29] 코드 설명 — MCP RAG 실행

**한 줄 요약:** 같은 질문을 MCP RAG로 풀어, 에이전트가 모은 정보와 최종 답을 출력합니다.

- `await rag_mcp.inference(query=question, search_results=[], query_time=query_time, topk=5)` : 웹 검색 결과는 빈 리스트로 전달
- `result["retrieved_results"]` : 에이전트가 KG에서 찾아 정리한 내용
- `result["answer"]` : 최종 답

> 🔰 # 13에서 본 `gold_answer` 와 비교해 보세요.

In [ ]:
# 29
### YOUR CODE HERE ###

result = await rag_mcp.inference(
    query=question,
    search_results=[],
    query_time=query_time,
    topk=5
)

print("Retrieved Results:\n", result["retrieved_results"])
print("------------------")
print("Final Answer:\n", result["answer"])

### 💡 [# 30] 코드 설명 — 3번 반복해 안정성 확인

**한 줄 요약:** MCP RAG로 같은 질문을 3번 풀어 답이 일관되게 나오는지 확인합니다.

- `for i in range(3):` 안에서 `await rag_mcp.inference(...)` 호출
- `rag_output['answer'].lower()` : 답을 소문자로 출력

> 🔰 # 17(기존 방식 3회)의 결과와 나란히 비교하면 MCP 도입의 효과를 확인할 수 있습니다. **직접 해 보기:** # 12의 평가 코드에서 `rag.inference(...)` 를 `await rag_mcp.inference(...)` 로 바꿔 CRAG 점수가 얼마나 오르는지 측정해 보세요.

In [ ]:
# 30
### YOUR CODE HERE ###

for i in range(3):
    rag_output = await rag_mcp.inference(
        query=question,
        search_results=[],
        query_time=query_time,
        topk=5
    )
    print(rag_output['answer'].lower())

In this notebook, we explored how MCP (Model Context Protocol) can be used to build a flexible and modular tool-augmented RAG pipeline.

While MCP allows LLMs to dynamically select and invoke tools using a standardized interface, real-world usage has revealed several practical limitations:

- **Context Overflow**: Tool outputs are inserted directly into the model's input context. If the result is too long, it may exceed the model’s token limit and lead to failure.
- **Limited Error Recovery**: When tool execution fails or exceeds limits, LLMs often cannot recover or retry unless explicitly guided to do so.
- **Debugging Difficulty**: Since tool selection and reasoning are tightly coupled inside the model, it is difficult to trace what went wrong without detailed logs or event streaming.
- **Latency and Reliability**: Each tool call requires a round-trip to the server. In multi-step workflows, this can introduce significant delay and failure points.
- **Loss of Developer Control**: Tool behavior is driven by the LLM’s interpretation of the prompt and available tools, making behavior harder to predict or constrain.

Understanding these trade-offs is key to using MCP effectively in real-world LLM applications.